# **Mountain NER Dataset Creation**

This notebook demonstrates the complete dataset creation pipeline used to build the training data for the Mountain Named Entity Recognition (NER) task.

The dataset was generated automatically using real mountain names collected from the Open Peaks project. The resulting dataset was later used to fine-tune a BERT-based NER model.

## Dataset Creation Pipeline

The dataset generation consists of three main stages:

1. Collect real mountain names from the Open Peaks dataset.
2. Generate positive BIO-annotated training examples.
3. Generate negative examples that do not contain mountain entities.

The overall pipeline is shown below:

```
Open Peaks
      │
      ▼
download_mountains.py
      │
      ▼
mountains.csv
      │
      ▼
generate_dataset.py
      │
      ▼
ner_dataset.json
```

## Step 1. Collect Mountain Names

Instead of creating mountain names manually, a publicly available dataset from the Open Peaks project was used.

The `download_mountains.py` script downloads the mountain index, extracts mountain names, removes duplicates, and stores the result as a CSV file.

Using real geographical names improves the realism of the generated dataset and reduces the chance of introducing artificial entities.

## Step 2. Generate Positive Examples

Positive training examples are created automatically by inserting real mountain names into predefined sentence templates.

Each generated sentence is then tokenized and annotated using the BIO tagging scheme.

This approach makes it possible to generate thousands of consistent training examples without manual annotation.

## Step 3. Generate Negative Examples

To reduce false positive predictions, additional negative examples were generated.

To improve the model's robustness, additional negative examples are generated automatically.

These sentences contain entities such as cities, people, organizations, companies, and technologies, but no mountain names.

All tokens are therefore labeled **O**, helping the model distinguish mountains from other entity types.

In [45]:
import pandas as pd

mountains = pd.read_csv("../data/raw/mountains.csv")

mountains.head(10)

,mountain
0,A'Mhaighdean
1,Abbott Butte
2,Acatenango
3,Achen Niyeu
4,Acho
5,Ackerlspitze
6,Aconcagua
7,Acotango
8,Adam's Peak
9,Adi Kailash


In [46]:
print(f"Total mountain names: {len(mountains)}")

Total mountain names: 2924


#### Load the Generated Dataset

In [47]:
import json

with open("../data/processed/ner_dataset.json", encoding="utf-8") as f:
    dataset = json.load(f)

print(f"Total samples: {len(dataset)}")

Total samples: 7000


### Missing Values

The generated dataset is expected to contain complete information for every sample.

The following check verifies that no fields are missing.

In [35]:
missing = sum(
    any(value is None for value in sample.values())
    for sample in dataset
)

print(f"Samples with missing values: {missing}")

Samples with missing values: 0


### Token–Label Consistency

Each token must have a corresponding BIO label.

The following check verifies that the number of tokens matches the number of labels in every sample.

In [33]:
invalid = sum(
    len(sample["tokens"]) != len(sample["ner_tags"])
    for sample in dataset
)

print(f"Samples with inconsistent token/label lengths: {invalid}")

Samples with inconsistent token/label lengths: 0


### Label Validation
The dataset is expected to contain only the predefined BIO labels:

- **0** — O
- **1** — B-MOUNTAIN
- **2** — I-MOUNTAIN

The following check verifies that no unexpected label IDs are present in the dataset.

In [44]:
valid_labels = {0, 1, 2}

invalid = sum(
    any(tag not in valid_labels for tag in sample["ner_tags"])
    for sample in dataset
)

print(f"Samples with invalid labels: {invalid}")

Samples with invalid labels: 0


## Dataset Format

Each sample is stored as a JSON object containing:

- unique sample ID;
- tokenized sentence;
- BIO labels.

In [19]:
dataset[1]

{'id': 1,
 'sentence': 'The trail leading to Pollock Mountain is considered challenging.',
 'tokens': ['The',
  'trail',
  'leading',
  'to',
  'Pollock',
  'Mountain',
  'is',
  'considered',
  'challenging',
  '.'],
 'ner_tags': [0, 0, 0, 0, 1, 2, 0, 0, 0, 0]}

## The dataset follows the standard BIO tagging scheme:
For better readability, the following dictionary converts the stored numeric labels into their corresponding BIO tags.

The label mapping is:                                          
- **B-MOUNTAIN** — beginning of a mountain name;    
- **I-MOUNTAIN** — continuation of the mountain name;    
- **O** — token outside any named entity.                  

In [22]:
LABELS = {
    0: "O",
    1: "B-MOUNTAIN",
    2: "I-MOUNTAIN"
}

### **Positive Example**
The following example illustrates how mountain entities are annotated using the BIO tagging scheme.

In [28]:
positive = next(
    sample for sample in dataset
    if any(tag != 0 for tag in sample["ner_tags"])
)

pd.DataFrame({
    "Token": positive["tokens"],
    "Label": [LABELS[tag] for tag in positive["ner_tags"]]
})

,Token,Label
0,The,O
1,trail,O
2,leading,O
3,to,O
4,Pollock,B-MOUNTAIN
5,Mountain,I-MOUNTAIN
6,is,O
7,considered,O
8,challenging,O
9,.,O


### **Negative Example**

Negative examples contain no mountain entities.

In [26]:
negative = next(
    sample for sample in dataset
    if all(tag == 0 for tag in sample["ner_tags"])
)

pd.DataFrame({
    "Token": negative["tokens"],
    "Label": [LABELS[tag] for tag in negative["ner_tags"]]
})

,Token,Label
0,Researchers,O
1,presented,O
2,a,O
3,paper,O
4,about,O
5,computer,O
6,vision,O
7,.,O


## **Dataset Statistics**

The following statistics summarize the generated dataset.

They include the number of collected mountain names, positive training examples, negative examples, and the total dataset size.

In [27]:
positive_count = sum(
    any(tag != 0 for tag in sample["ner_tags"])
    for sample in dataset
)

negative_count = len(dataset) - positive_count

print(f"Mountain names      : {len(mountains)}")
print(f"Positive examples   : {positive_count}")
print(f"Negative examples   : {negative_count}")
print(f"Total examples      : {len(dataset)}")

Mountain names      : 2924
Positive examples   : 5000
Negative examples   : 2000
Total examples      : 7000


## Advantages

- Uses real mountain names collected from the Open Peaks dataset.
- Automatically generates BIO annotations.
- Includes both positive and negative examples.
- Easy to extend by adding new templates or mountain names.
- Fully reproducible dataset generation pipeline.

## Limitations

- The dataset contains only mountain names and does not include other geographical entities.
- Most training sentences are automatically generated rather than collected from real-world documents.
- The validation set follows the same synthetic generation process as the training data.
- Linguistic diversity is limited by the predefined sentence templates.

## **Conclusion**

This notebook demonstrates the complete dataset creation workflow used in the project.

The final dataset contains **7000 BIO-annotated examples**, including **5000 positive** and **2000 negative** samples, generated from **2924 real mountain names**.

The resulting dataset was used to successfully fine-tune a BERT-based Named Entity Recognition model for mountain name detection.